<a href="https://colab.research.google.com/github/samuelsambow12345-source/William-Sambow/blob/main/Test_Tabnet_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Test Tabnet

In [ ]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

!pip install pytorch-tabnet
from pytorch_tabnet.tab_model import TabNetClassifier

# Import SMOTE
from imblearn.over_sampling import SMOTE

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

In [ ]:
file = "PENGGABUNGAN DATA 3.xlsx"

df = pd.read_excel(
    file,
    sheet_name="Dataset All Clean"
)

print("Ukuran dataset:", df.shape)
print(df.head())


Ukuran dataset: (8796, 57)
         Company Common Name                                   Exchange Name  \
0  Astra Agro Lestari Tbk PT  Indonesia Stock Exchange (formerly Jakarta SE)   
1  Astra Agro Lestari Tbk PT  Indonesia Stock Exchange (formerly Jakarta SE)   
2  Astra Agro Lestari Tbk PT  Indonesia Stock Exchange (formerly Jakarta SE)   
3  Astra Agro Lestari Tbk PT  Indonesia Stock Exchange (formerly Jakarta SE)   
4  Astra Agro Lestari Tbk PT  Indonesia Stock Exchange (formerly Jakarta SE)   

   GICS Sector Name Exchange Ticker Short Exchange Name  Year        ROA  \
0  Consumer Staples            AALI                 IDX  2005  24.839745   
1  Consumer Staples            AALI                 IDX  2006  24.340594   
2  Consumer Staples            AALI                 IDX  2007  46.099901   
3  Consumer Staples            AALI                 IDX  2008  25.402751   
4  Consumer Staples            AALI                 IDX  2009  13.791866   

         ROE    EBITTA       DE  ..

In [ ]:
print("\nDistribusi Label:")
print(df["Label"].value_counts())



Distribusi Label:
Label
0    4760
1    4036
Name: count, dtype: int64


In [ ]:
X = df.drop(columns=[
    "ROA",
    "ROE",
    "EBITTA", "DE", "DA", "ICR", "CR", "CA", "WCTA",
    "OCF", "WC", "TA", "NIBT", "REV", "NI", "TD",
    "CA","CL","EBITDA M","EBIT M", "RETA"
])

y = df["Label"]


In [ ]:
print("\nTipe data X:")
print(X.dtypes)


Tipe data X:
Company Common Name     object
Exchange Name           object
GICS Sector Name        object
Exchange Ticker         object
Short Exchange Name     object
Year                     int64
CA.1                   float64
CP                     float64
CP NEXT                float64
Label                    int64
Target_Label            object
Unnamed: 31            float64
Unnamed: 32            float64
DATA KOSONG             object
Unnamed: 34             object
ROA.1                  float64
ROE.1                  float64
EBITTA.1               float64
DE.1                   float64
DA.1                   float64
ICR.1                  float64
CR.1                   float64
CA.2                   float64
WCTA.1                 float64
OCF.1                  float64
WC.1                   float64
TA.1                   float64
NIBT.1                 float64
REV.1                  float64
NI.1                   float64
TD.1                   float64
CA.3                   fl

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train
)

In [ ]:
# First, identify all columns that are not suitable for median imputation (object types and the 'Label' target column).
# The 'X_train' at this point has 28 columns, as per kernel state.
cols_to_drop_from_imputation = X_train.select_dtypes(include=['object']).columns.tolist()
if 'Label' in X_train.columns:
    cols_to_drop_from_imputation.append('Label')

# Explicitly drop these columns from all splits *before* imputation.
# This ensures that the imputer only sees numerical columns.
X_train_numeric = X_train.drop(columns=cols_to_drop_from_imputation, errors='ignore')
X_valid_numeric = X_valid.drop(columns=cols_to_drop_from_imputation, errors='ignore')
X_test_numeric = X_test.drop(columns=cols_to_drop_from_imputation, errors='ignore')

# Initialize imputer
imputer = SimpleImputer(strategy="median")

# Fit imputer on training data and transform all datasets.
# The imputer will return a NumPy array.
imputed_X_train_array = imputer.fit_transform(X_train_numeric)
imputed_X_valid_array = imputer.transform(X_valid_numeric)
imputed_X_test_array = imputer.transform(X_test_numeric)

# Get the feature names that the imputer actually used (i.e., not skipped).
# This will correctly handle columns like 'Unnamed: 31' and 'Unnamed: 32' if they were entirely NaN.
imputed_feature_names = imputer.get_feature_names_out(X_train_numeric.columns)

# Reconstruct DataFrames with correct column names
X_train = pd.DataFrame(
    imputed_X_train_array,
    columns=imputed_feature_names,
    index=X_train.index
)

X_valid = pd.DataFrame(
    imputed_X_valid_array,
    columns=imputed_feature_names,
    index=X_valid.index
)

X_test = pd.DataFrame(
    imputed_X_test_array,
    columns=imputed_feature_names,
    index=X_test.index
)

/usr/local/lib/python3.13/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['Unnamed: 31' 'Unnamed: 32']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['Unnamed: 31' 'Unnamed: 32']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['Unnamed: 31' 'Unnamed: 32']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


In [ ]:
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print("\nLabel sebelum SMOTE:")
print(y_train.value_counts())

print("\nLabel setelah SMOTE:")
print(pd.Series(y_train_smote).value_counts())



Label sebelum SMOTE:
Label
0    3046
1    2582
Name: count, dtype: int64

Label setelah SMOTE:
Label
0    3046
1    3046
Name: count, dtype: int64


In [ ]:
X_train_smote = X_train_smote.to_numpy()
X_valid = X_valid.to_numpy()
X_test = X_test.to_numpy()

y_train_smote = np.asarray(y_train_smote)
y_valid = np.asarray(y_valid)
y_test = np.asarray(y_test)



In [ ]:

clf = TabNetClassifier(
    seed=42
)

/usr/local/lib/python3.13/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


In [ ]:
clf.fit(
    X_train_smote,
    y_train_smote,

    eval_set=[
        (X_train_smote, y_train_smote),
        (X_valid, y_valid)
    ],

    eval_name=[
        "train",
        "valid"
    ],

    eval_metric=[
        "auc"
    ],

    max_epochs=100,
    patience=15,

    batch_size=256,
    virtual_batch_size=128
)

epoch 0  | loss: 0.76573 | train_auc: 0.49775 | valid_auc: 0.48081 |  0:00:01s
epoch 1  | loss: 0.69062 | train_auc: 0.50195 | valid_auc: 0.50103 |  0:00:01s
epoch 2  | loss: 0.67441 | train_auc: 0.49984 | valid_auc: 0.49815 |  0:00:02s
epoch 3  | loss: 0.66996 | train_auc: 0.61819 | valid_auc: 0.6317  |  0:00:03s
epoch 4  | loss: 0.61602 | train_auc: 0.61462 | valid_auc: 0.61899 |  0:00:05s
epoch 5  | loss: 0.55184 | train_auc: 0.56824 | valid_auc: 0.57924 |  0:00:06s
epoch 6  | loss: 0.45824 | train_auc: 0.50016 | valid_auc: 0.50077 |  0:00:06s
epoch 7  | loss: 0.41642 | train_auc: 0.49935 | valid_auc: 0.50012 |  0:00:07s
epoch 8  | loss: 0.36676 | train_auc: 0.54181 | valid_auc: 0.53122 |  0:00:08s
epoch 9  | loss: 0.36511 | train_auc: 0.54038 | valid_auc: 0.53135 |  0:00:09s
epoch 10 | loss: 0.3414  | train_auc: 0.62703 | valid_auc: 0.63331 |  0:00:10s
epoch 11 | loss: 0.30935 | train_auc: 0.63446 | valid_auc: 0.64103 |  0:00:11s
epoch 12 | loss: 0.24111 | train_auc: 0.62714 | vali

/usr/local/lib/python3.13/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


In [ ]:
preds = clf.predict(X_test)

preds_proba = clf.predict_proba(X_test)


In [ ]:
accuracy = accuracy_score(
    y_test,
    preds
)

precision = precision_score(
    y_test,
    preds,
    zero_division=0
)

recall = recall_score(
    y_test,
    preds,
    zero_division=0
)

f1 = f1_score(
    y_test,
    preds,
    zero_division=0
)
auc = roc_auc_score(
    y_test,
    preds_proba[:, 1]
)

In [ ]:
print("\n====================================")
print("HASIL TABNET")
print("====================================")

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")
print(f"AUC-ROC   : {auc:.4f}")





HASIL TABNET
Accuracy  : 0.5409
Precision : 0.0000
Recall    : 0.0000
F1-Score  : 0.0000
AUC-ROC   : 0.9543


In [ ]:
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, preds))



Confusion Matrix:
[[952   0]
 [808   0]]


In [ ]:
print("\nClassification Report:")
print(
    classification_report(
        y_test,
        preds,
        zero_division=0
    )
)


Classification Report:
              precision    recall  f1-score   support

           0       0.54      1.00      0.70       952
           1       0.00      0.00      0.00       808

    accuracy                           0.54      1760
   macro avg       0.27      0.50      0.35      1760
weighted avg       0.29      0.54      0.38      1760



In [ ]:
git init
git add README.md
git commit -m "first commit"
git branch -M main
git remote add origin https://github.com/samuelsambow12345-source/William-Samuel.git
git push -u origin main